# ResNet101 Growth-Stage Classifier — U-Net-Predicted-Mask Images (Google Colab)

Trains a ResNet101 on tomato images masked **by U-Net's own predicted masks** (not
ground-truth COCO polygons), then compares against the existing ResNet50 trained on
ground-truth-masked images (`checkpoints_masked/resnet50_masked_best.pt`, val accuracy
97.78%, used in the 89.4%/85.6% single-split/cross-validated fusion pipeline).

**Important — read before running:** this notebook varies **two things at once** relative
to the existing ResNet50 masked run: the backbone (ResNet50→101) *and* the mask source
(ground-truth polygons → U-Net predictions). It is **not** a clean architecture-only
ablation — if a straight backbone comparison is also needed, that requires a separate run
using the ground-truth-masked dataset instead (`tomato final dataset_masked/`). Framed
correctly in a thesis, this is its own experiment: *does training the classifier on the
same kind of (imperfect) masks it sees at pipeline inference time improve full-pipeline
accuracy*, independent of which backbone is used.

Everything else — optimizer, LR, epochs, batch size, augmentation, seed — matches the
ResNet50 masked run exactly.

## Before running this notebook

1. **Upload `unet_tomato_final.pth`** directly into this Colab session's Files panel (left
   sidebar, upload icon) — it needs to be re-uploaded each time you start a fresh runtime,
   since local Colab storage doesn't persist across sessions.
2. **Your raw dataset** should already be in Google Drive as a folder (zipped or not) —
   the config cell resolves it directly by its Drive folder ID, so no need to know its exact
   path. If you're linking a different folder later, just swap in its ID from the folder's
   share URL (`drive.google.com/drive/folders/<ID>`).
3. **Runtime → Change runtime type → GPU** before running.
4. Run cells top to bottom — the dataset-resolution cell (Section 1) mounts Drive, locates
   the folder, and stages it to local Colab disk automatically.


## 1. Colab setup

In [ ]:
from google.colab import drive
drive.mount("/content/drive")


In [ ]:
import torch
print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    raise RuntimeError("No GPU detected -- go to Runtime > Change runtime type > GPU, then Runtime > Restart and run all.")


In [ ]:
import os

# --- Dataset: your shared Google Drive folder, resolved by ID (works whether it's your
# own folder or one shared with you, no need to know its exact path in Drive) ---
DATASET_FOLDER_ID = "1uXlpfWuuCAWo6J6Fr0NirlsEXyg9GkQe"

# U-Net checkpoint: uploaded directly into this Colab session's local storage
DRIVE_UNET_CKPT = "/content/unet_tomato_final.pth"

LOCAL_RAW_DIR = "/content/data_raw"
LOCAL_MASKED_DIR = "/content/data_unet_masked"
os.makedirs(LOCAL_RAW_DIR, exist_ok=True)
os.makedirs(LOCAL_MASKED_DIR, exist_ok=True)

assert os.path.exists(DRIVE_UNET_CKPT), (
    f"U-Net checkpoint not found at {DRIVE_UNET_CKPT} -- upload unet_tomato_final.pth "
    "to the Colab Files panel (left sidebar) first."
)

DATASET_DRIVE_PATH = f"/content/drive/.shortcut-targets-by-id/{DATASET_FOLDER_ID}"
assert os.path.isdir(DATASET_DRIVE_PATH), (
    f"Could not find the dataset at {DATASET_DRIVE_PATH}. Make sure Drive is mounted (cell above), "
    "and if this folder was shared with you (not owned by you), right-click it at drive.google.com "
    "and choose 'Add shortcut to Drive' first, then retry."
)
print("found dataset folder, contents:")
!ls "{DATASET_DRIVE_PATH}"

# Copy/unzip to local Colab disk -- reading thousands of small files directly off a mounted
# Drive (FUSE) is much slower than local disk, which matters during mask generation + training.
zip_files = [f for f in os.listdir(DATASET_DRIVE_PATH) if f.lower().endswith(".zip")]
if zip_files:
    zip_path = os.path.join(DATASET_DRIVE_PATH, zip_files[0])
    print(f"found zip: {zip_files[0]}, unzipping...")
    !unzip -q -n "{zip_path}" -d "{LOCAL_RAW_DIR}"
else:
    print("no zip found -- copying raw folder contents directly (this may take a few minutes)...")
    !cp -r "{DATASET_DRIVE_PATH}"/. "{LOCAL_RAW_DIR}"/

print("dataset staged at", LOCAL_RAW_DIR)
!ls "{LOCAL_RAW_DIR}"


## 2. Config

In [ ]:
import copy
import time
import random

import numpy as np
import cv2
import matplotlib.pyplot as plt
import seaborn as sns

import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from PIL import Image
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Locate the real dataset root by looking for a folder containing train/valid/test
# subdirectories -- robust regardless of what the Drive folder or zip is actually named.
def _find_dataset_root(base):
    for root, dirs, _files in os.walk(base):
        if "train" in dirs and "valid" in dirs and "test" in dirs:
            return root
    raise FileNotFoundError(
        f"Could not find a folder with train/valid/test subdirectories under {base}. "
        "Check the dataset was staged correctly in the cell above."
    )

DATA_DIR_RAW = _find_dataset_root(LOCAL_RAW_DIR)

DRIVE_OUTPUT_DIR = "/content/drive/MyDrive/tomato_research/resnet101_unet_masked"
CHECKPOINT_DIR = os.path.join(DRIVE_OUTPUT_DIR, "checkpoints")
EXPORT_DIR = os.path.join(DRIVE_OUTPUT_DIR, "export")
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(EXPORT_DIR, exist_ok=True)

STAGE_NAMES = ["developing", "flowering", "fruiting", "seeding"]
SPLITS = ["train", "valid", "test"]

# --- hyperparameters -- identical to the ResNet50 masked run, only backbone + mask source change ---
IMAGE_SIZE = 224
BATCH_SIZE = 32
NUM_EPOCHS = 20
LEARNING_RATE = 1e-4
NUM_WORKERS = 2
FREEZE_BACKBONE = False
MASK_THRESHOLD = 0.5

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

print("raw dataset root:", DATA_DIR_RAW)
print("outputs will be saved under:", DRIVE_OUTPUT_DIR)


## 3. Load U-Net and generate masked images

U-Net is defined here unchanged from `export/unet_lpf_extractor.py` (no dependency on the
segmentation project's training code, just the exported architecture + checkpoint). For
each raw image, applies the same crop-aligned frame the classifier will train on
(`Resize(256)→CenterCrop(224)`) *before* running U-Net — this keeps the predicted mask
geometrically aligned with the classifier's input, the same fix that took the full pipeline
from 78.75%→89.4% in the segmentation project. The masked result is saved once per image
as a 224×224 JPEG, mirroring the raw dataset's `{split}/{stage}/*.jpg` layout.

In [ ]:
class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1), nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1), nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
        )
    def forward(self, x): return self.block(x)


class UNet(nn.Module):
    def __init__(self, in_channels=3, out_channels=1, base_channels=32):
        super().__init__()
        c = base_channels
        self.enc1 = DoubleConv(in_channels, c); self.enc2 = DoubleConv(c, c * 2)
        self.enc3 = DoubleConv(c * 2, c * 4); self.enc4 = DoubleConv(c * 4, c * 8)
        self.pool = nn.MaxPool2d(2)
        self.bottleneck = DoubleConv(c * 8, c * 16)
        self.up4 = nn.ConvTranspose2d(c * 16, c * 8, 2, stride=2); self.dec4 = DoubleConv(c * 16, c * 8)
        self.up3 = nn.ConvTranspose2d(c * 8, c * 4, 2, stride=2); self.dec3 = DoubleConv(c * 8, c * 4)
        self.up2 = nn.ConvTranspose2d(c * 4, c * 2, 2, stride=2); self.dec2 = DoubleConv(c * 4, c * 2)
        self.up1 = nn.ConvTranspose2d(c * 2, c, 2, stride=2); self.dec1 = DoubleConv(c * 2, c)
        self.out_conv = nn.Conv2d(c, out_channels, 1)

    def forward(self, x):
        e1 = self.enc1(x); e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2)); e4 = self.enc4(self.pool(e3))
        b = self.bottleneck(self.pool(e4))
        d4 = self.dec4(torch.cat([self.up4(b), e4], dim=1))
        d3 = self.dec3(torch.cat([self.up3(d4), e3], dim=1))
        d2 = self.dec2(torch.cat([self.up2(d3), e2], dim=1))
        d1 = self.dec1(torch.cat([self.up1(d2), e1], dim=1))
        return self.out_conv(d1)


unet_model = UNet(base_channels=32).to(device)
unet_state = torch.load(DRIVE_UNET_CKPT, map_location=device, weights_only=True)
unet_model.load_state_dict(unet_state)
unet_model.eval()
print("U-Net loaded, params:", sum(p.numel() for p in unet_model.parameters()))


In [ ]:
crop_transform = transforms.Compose([
    transforms.Resize(int(IMAGE_SIZE * 1.14)),
    transforms.CenterCrop(IMAGE_SIZE),
])
to_tensor = transforms.ToTensor()
normalize = transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)


@torch.no_grad()
def mask_and_save(src_path, dst_path, threshold=MASK_THRESHOLD):
    img = Image.open(src_path).convert("RGB")
    img_cropped = crop_transform(img)              # 224x224, aspect-preserving, classifier's own frame
    raw01 = to_tensor(img_cropped)                  # 0-1 scale CHW

    normed = normalize(raw01).unsqueeze(0).to(device)
    probs = torch.sigmoid(unet_model(normed))
    pred_mask = (probs > threshold).float()

    masked = (raw01.unsqueeze(0).to(device) * pred_mask).squeeze(0).cpu()
    masked_img = transforms.ToPILImage()(masked)
    os.makedirs(os.path.dirname(dst_path), exist_ok=True)
    masked_img.save(dst_path, quality=95)


n_done = 0
for split in SPLITS:
    for stage in STAGE_NAMES:
        src_dir = os.path.join(DATA_DIR_RAW, split, stage)
        if not os.path.isdir(src_dir):
            print(f"WARNING: missing {src_dir}, skipping")
            continue
        fnames = sorted(f for f in os.listdir(src_dir) if f.lower().endswith((".jpg", ".jpeg", ".png")))
        for fname in tqdm(fnames, desc=f"{split}/{stage}", leave=False):
            src_path = os.path.join(src_dir, fname)
            dst_path = os.path.join(LOCAL_MASKED_DIR, split, stage, fname)
            mask_and_save(src_path, dst_path)
            n_done += 1

print(f"generated {n_done} U-Net-masked images under {LOCAL_MASKED_DIR}")


## 4. Data loading

Same split-first/class-inside layout (`root/<split>/<class>/*.jpg`) as the ground-truth
masked dataset, just pointed at the U-Net-masked images generated above. Note images here
are already fixed at 224×224 (masked at that resolution in step 3), so
`RandomResizedCrop`'s effective augmentation range is more limited than when it operates on
full-resolution source images — kept anyway for consistency with the ResNet50 masked run's
transform pipeline.

In [ ]:
class SplitFirstFolderDataset(Dataset):
    """Reads root/<split>/<class>/*.jpg."""

    IMG_EXTS = (".jpg", ".jpeg", ".png", ".bmp")

    def __init__(self, root, split, transform, classes=None):
        self.transform = transform
        split_root = os.path.join(root, split)
        self.classes = classes or sorted(
            d for d in os.listdir(split_root) if os.path.isdir(os.path.join(split_root, d))
        )
        self.samples = []
        for label_idx, cls in enumerate(self.classes):
            class_dir = os.path.join(split_root, cls)
            for fname in sorted(os.listdir(class_dir)):
                if fname.lower().endswith(self.IMG_EXTS):
                    self.samples.append((os.path.join(class_dir, fname), label_idx))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = Image.open(path).convert("RGB")
        return self.transform(img), label


In [ ]:
train_transform = transforms.Compose([
    transforms.RandomResizedCrop(IMAGE_SIZE, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

eval_transform = transforms.Compose([
    transforms.Resize(int(IMAGE_SIZE * 1.14)),
    transforms.CenterCrop(IMAGE_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

train_ds = SplitFirstFolderDataset(LOCAL_MASKED_DIR, "train", train_transform)
class_names = train_ds.classes
num_classes = len(class_names)
val_ds = SplitFirstFolderDataset(LOCAL_MASKED_DIR, "valid", eval_transform, classes=class_names)
test_ds = SplitFirstFolderDataset(LOCAL_MASKED_DIR, "test", eval_transform, classes=class_names)

print("classes:", class_names)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS,
                           pin_memory=True, persistent_workers=(NUM_WORKERS > 0))
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS,
                         pin_memory=True, persistent_workers=(NUM_WORKERS > 0))
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS,
                          pin_memory=True, persistent_workers=(NUM_WORKERS > 0))

print(f"train: {len(train_ds)} | valid: {len(val_ds)} | test: {len(test_ds)}")


## 5. Model — ResNet101

ImageNet-pretrained ResNet101, final fully-connected layer replaced with
`nn.Linear(2048, 4)`, full network fine-tuned.

In [ ]:
def build_model():
    m = models.resnet101(weights=models.ResNet101_Weights.IMAGENET1K_V2)
    if FREEZE_BACKBONE:
        for p in m.parameters():
            p.requires_grad = False
    m.fc = nn.Linear(m.fc.in_features, num_classes)
    return m.to(device)

model = build_model()

criterion = nn.CrossEntropyLoss()
params_to_update = [p for p in model.parameters() if p.requires_grad]
optimizer = optim.Adam(params_to_update, lr=LEARNING_RATE)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=2)

print(f"trainable params: {sum(p.numel() for p in params_to_update):,}")


## 6. Training loop

In [ ]:
def run_epoch(model, loader, criterion, optimizer=None):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()

    running_loss, running_correct, total = 0.0, 0, 0
    with torch.set_grad_enabled(is_train):
        for inputs, labels in loader:
            inputs, labels = inputs.to(device), labels.to(device)

            if is_train:
                optimizer.zero_grad()

            outputs = model(inputs)
            loss = criterion(outputs, labels)

            if is_train:
                loss.backward()
                optimizer.step()

            preds = outputs.argmax(dim=1)
            running_loss += loss.item() * inputs.size(0)
            running_correct += (preds == labels).sum().item()
            total += inputs.size(0)

    return running_loss / total, running_correct / total


In [ ]:
history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
best_val_acc = 0.0
best_model_wts = copy.deepcopy(model.state_dict())
best_ckpt_path = os.path.join(CHECKPOINT_DIR, "resnet101_unet_masked_best.pt")

start = time.time()
for epoch in range(1, NUM_EPOCHS + 1):
    train_loss, train_acc = run_epoch(model, train_loader, criterion, optimizer)
    val_loss, val_acc = run_epoch(model, val_loader, criterion, optimizer=None)
    scheduler.step(val_loss)

    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_model_wts = copy.deepcopy(model.state_dict())
        torch.save({
            "model_state_dict": best_model_wts,
            "class_names": class_names,
            "epoch": epoch,
            "val_acc": val_acc,
        }, best_ckpt_path)

    print(f"Epoch {epoch:02d}/{NUM_EPOCHS} | "
          f"train_loss {train_loss:.4f} train_acc {train_acc:.4f} | "
          f"val_loss {val_loss:.4f} val_acc {val_acc:.4f}")

elapsed = time.time() - start
print(f"\nTraining complete in {elapsed // 60:.0f}m {elapsed % 60:.0f}s")
print(f"Best val accuracy: {best_val_acc:.4f} (saved to {best_ckpt_path})")

model.load_state_dict(best_model_wts)
model.eval()


## 7. Training curves

In [ ]:
epochs_range = range(1, len(history["train_loss"]) + 1)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(epochs_range, history["train_loss"], label="train")
axes[0].plot(epochs_range, history["val_loss"], label="val")
axes[0].set_title("Loss"); axes[0].set_xlabel("epoch"); axes[0].legend()

axes[1].plot(epochs_range, history["train_acc"], label="train")
axes[1].plot(epochs_range, history["val_acc"], label="val")
axes[1].set_title("Accuracy"); axes[1].set_xlabel("epoch"); axes[1].legend()

plt.tight_layout()
plt.savefig(os.path.join(DRIVE_OUTPUT_DIR, "training_curves.png"), dpi=150)
plt.show()


## 8. Test-set evaluation

In [ ]:
@torch.no_grad()
def evaluate(eval_model, loader):
    eval_model.eval()
    all_preds, all_labels = [], []
    for inputs, labels in loader:
        inputs = inputs.to(device)
        outputs = eval_model(inputs)
        preds = outputs.argmax(dim=1).cpu().numpy()
        all_preds.extend(preds.tolist())
        all_labels.extend(labels.numpy().tolist())
    acc = accuracy_score(all_labels, all_preds)
    report = classification_report(all_labels, all_preds, target_names=class_names, digits=4)
    cm = confusion_matrix(all_labels, all_preds)
    return acc, report, cm, all_labels, all_preds

test_acc, test_report, test_cm, _, _ = evaluate(model, test_loader)
print(f"Test accuracy (on U-Net-masked test images): {test_acc:.4f}\n")
print(test_report)

plt.figure(figsize=(5.5, 4.5))
sns.heatmap(test_cm, annot=True, fmt="d", cmap="Blues", xticklabels=class_names, yticklabels=class_names)
plt.xlabel("predicted"); plt.ylabel("true")
plt.title(f"ResNet101 (U-Net-masked), test acc={test_acc:.4f}")
plt.tight_layout()
plt.savefig(os.path.join(DRIVE_OUTPUT_DIR, "test_confusion_matrix.png"), dpi=150)
plt.show()


**Note:** this test accuracy is measured on images masked by U-Net at their *original*
resolution-then-cropped frame — not the same thing as running the full raw-image→U-Net→classify
pipeline end to end (Section 9 below does that). This section only tells you how good the
classifier is at recognizing growth stage given a mask it was trained on the same way it
receives at inference.

## 9. Full pipeline evaluation (optional, for direct comparison to 89.4%/85.6%)

Re-runs U-Net inference at eval time (rather than reusing the pre-masked images from
Section 3) and classifies with this notebook's ResNet101 — the same "segment → mask →
classify" pipeline evaluated in `unet_resnet_fusion_pipeline.ipynb`, so this number is
directly comparable to that project's 89.4% single-split / 85.6% cross-validated results.

In [ ]:
@torch.no_grad()
def predict_growth_stage(image_path, threshold=MASK_THRESHOLD):
    img = Image.open(image_path).convert("RGB")
    img_cropped = crop_transform(img)
    raw01 = to_tensor(img_cropped)

    normed = normalize(raw01).unsqueeze(0).to(device)
    probs = torch.sigmoid(unet_model(normed))
    pred_mask = (probs > threshold).float()

    masked_raw = raw01.unsqueeze(0).to(device) * pred_mask
    masked_norm = normalize(masked_raw.squeeze(0)).unsqueeze(0)

    logits = model(masked_norm)
    pred_idx = torch.softmax(logits, dim=1)[0].argmax().item()
    return class_names[pred_idx]


pipeline_preds, pipeline_labels = [], []
for stage in STAGE_NAMES:
    test_dir = os.path.join(DATA_DIR_RAW, "test", stage)
    for fname in sorted(os.listdir(test_dir)):
        if not fname.lower().endswith((".jpg", ".jpeg", ".png")):
            continue
        pred = predict_growth_stage(os.path.join(test_dir, fname))
        pipeline_preds.append(pred)
        pipeline_labels.append(stage)

pipeline_acc = accuracy_score(pipeline_labels, pipeline_preds)
print(f"Full pipeline (raw image -> U-Net -> ResNet101) test accuracy: {pipeline_acc:.4f}")
print(f"(compare to: U-Net + ResNet50 single-split 0.8938, cross-validated mean 0.8562 +/- 0.0238)\n")
print(classification_report(pipeline_labels, pipeline_preds, labels=STAGE_NAMES, target_names=STAGE_NAMES, digits=4))


## 10. Export

In [ ]:
import json
import datetime

torch.save(model.state_dict(), os.path.join(EXPORT_DIR, "resnet101_unet_masked.pth"))

metadata = {
    "architecture": "resnet101",
    "modifications": "torchvision resnet101 (ImageNet1K_V2 pretrained), fc replaced with "
                      "nn.Linear(2048, 4), full network fine-tuned.",
    "mask_source": "U-Net-predicted masks (export/unet_tomato_final.pth), NOT ground-truth COCO "
                    "polygon masks -- not directly comparable to checkpoints_masked/resnet50_masked_best.pt "
                    "as a pure backbone ablation; two variables differ (architecture and mask source).",
    "input_size": [IMAGE_SIZE, IMAGE_SIZE, 3],
    "num_classes": num_classes,
    "class_names": class_names,
    "normalization": {"mean": IMAGENET_MEAN, "std": IMAGENET_STD},
    "training": {
        "epochs_run": NUM_EPOCHS,
        "optimizer": "Adam",
        "learning_rate": LEARNING_RATE,
        "batch_size": BATCH_SIZE,
        "seed": SEED,
        "train_data": "raw tomato final dataset/train, masked on-the-fly by U-Net at crop-aligned 224x224",
    },
    "val_accuracy_at_best_epoch": best_val_acc,
    "test_accuracy_on_premasked_images": test_acc,
    "full_pipeline_test_accuracy": pipeline_acc,
    "training_date": datetime.date.today().isoformat(),
    "framework": {"torch": torch.__version__},
    "source_checkpoint": "checkpoints/resnet101_unet_masked_best.pt",
    "comparison_note": "Same optimizer/LR/epochs/batch size/seed as checkpoints_masked/resnet50_masked_best.pt "
                        "(val_acc 0.9778), but trained on U-Net-predicted masks rather than ground truth -- "
                        "compare full_pipeline_test_accuracy against the U-Net+ResNet50 pipeline's 0.8938 "
                        "single-split / 0.8562 cross-validated accuracy, not against resnet50_masked_best.pt's "
                        "own reported numbers directly.",
}

with open(os.path.join(EXPORT_DIR, "metadata.json"), "w") as f:
    json.dump(metadata, f, indent=2)

print("exported to", EXPORT_DIR)
print(json.dumps(metadata, indent=2))


### Bringing this back into your local project

Everything under `DRIVE_OUTPUT_DIR` (`MyDrive/tomato_research/resnet101_unet_masked/`)
persists on Drive after the Colab runtime disconnects. Download it to bring
`resnet101_unet_masked_best.pt`, `export/`, and the plots back into your local
`Resnet50test` project.
